# Split GSM8K into easy/hard for the e3 curriculum

This notebook turns the per-problem difficulty eval (strict scorer, 512-token budget) into the four training parquets used by Track C (clean) and Track D (mixed):

- `train_easy_clean.parquet`, `train_hard_clean.parquet`
- `train_easy_mixed.parquet`, `train_hard_mixed.parquet`

It also lets you **spot-check** sample easy vs hard questions alongside the base model's generated answers.

Prerequisite: run the train-split difficulty eval first:
```bash
modal run modal_eval_general.py --dataset gsm8k --model qwen --split train \
    --scorer gsm8k_strict --max-response-length 512 \
    --output-dir /data/e3_gsm8k --output-tag train_strict_l512
```

## 1. Configuration

In [ ]:
import os
import sys
import subprocess
import random

import pandas as pd
import datasets

# Repo root so we can reuse the gsm8k_padded helpers.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

VOLUME = "e3-generation-vol"
REMOTE_DIR = "e3_gsm8k"                      # /data/e3_gsm8k on the Volume
EVAL_TAG = "train_strict_l512"
PER_PROBLEM_CSV = f"per_problem_gsm8k_qwen_{EVAL_TAG}.csv"
OUTPUTS_PARQUET = f"gsm8k_qwen_{EVAL_TAG}_outputs.parquet"

# Local working dir for downloaded artifacts + generated parquets.
LOCAL_DIR = os.path.join(REPO_ROOT, "scripts", "e3-grpo-gsm8k", "data_e3_gsm8k")
os.makedirs(LOCAL_DIR, exist_ok=True)

EASY_FRACTION = 0.70   # ~70% easiest problems -> easy set
SEED = 42
print(f"REPO_ROOT={REPO_ROOT}\nLOCAL_DIR={LOCAL_DIR}")

## 2. Download eval artifacts from the Modal Volume

In [ ]:
def modal_get(remote_name: str):
    """Download a file from the Volume into LOCAL_DIR (overwrites)."""
    remote_path = f"{REMOTE_DIR}/{remote_name}"
    local_path = os.path.join(LOCAL_DIR, remote_name)
    if os.path.exists(local_path):
        os.remove(local_path)
    cmd = ["modal", "volume", "get", VOLUME, remote_path, local_path]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)
    return local_path

per_problem_path = modal_get(PER_PROBLEM_CSV)
# The outputs parquet (with base-model responses) is only needed for spot-checking.
try:
    outputs_path = modal_get(OUTPUTS_PARQUET)
except subprocess.CalledProcessError:
    outputs_path = None
    print("[warn] outputs parquet not found; spot-check of model answers will be skipped")

## 3. Assign easy/hard by accuracy

In [ ]:
per_problem = pd.read_csv(per_problem_path)
print(f"loaded {len(per_problem)} problems")
print(per_problem["accuracy"].describe())

# Sort by accuracy descending: highest accuracy = easiest.
ranked = per_problem.sort_values("accuracy", ascending=False).reset_index(drop=True)
n_easy = int(round(len(ranked) * EASY_FRACTION))
easy_idx = set(ranked.loc[:n_easy - 1, "problem_idx"].astype(int))
hard_idx = set(ranked.loc[n_easy:, "problem_idx"].astype(int))
print(f"easy={len(easy_idx)}  hard={len(hard_idx)}")

# Accuracy threshold at the split boundary (for reporting).
threshold = ranked.loc[n_easy - 1, "accuracy"] if n_easy > 0 else None
print(f"boundary accuracy ~ {threshold}")

## 4. Build the parquets (reusing gsm8k_padded helpers)

In [ ]:
from examples.data_preprocess.gsm8k_padded import (
    build_row,
    extract_solution,
    TRIVIA_FACTS,
)

ds = datasets.load_dataset("openai/gsm8k", "main")
train_raw = ds["train"]
test_raw = ds["test"]
print(f"gsm8k train={len(train_raw)}  test={len(test_raw)}")
assert len(train_raw) == len(per_problem), (
    "per-problem rows must align 1:1 with the gsm8k train split order"
)

rng = random.Random(SEED)

def build_clean_rows(indices):
    rows = []
    for new_i, orig_i in enumerate(sorted(indices)):
        ex = train_raw[int(orig_i)]
        q, a = ex["question"], ex["answer"]
        gt = extract_solution(a)
        rows.append(build_row(q, gt, "train", new_i, a, q))
    return rows

def build_mixed_rows(indices):
    # Clean originals first, then one trivia-padded clone of each (Track-D style).
    clean = build_clean_rows(indices)
    M = len(clean)
    padded = []
    for k, orig_i in enumerate(sorted(indices)):
        ex = train_raw[int(orig_i)]
        q, a = ex["question"], ex["answer"]
        gt = extract_solution(a)
        fact = rng.choice(TRIVIA_FACTS)
        padded_q = f"{fact} {q}"
        padded.append(build_row(padded_q, gt, "train", M + k, a, padded_q))
    return clean + padded

def write_parquet(rows, name):
    path = os.path.join(LOCAL_DIR, name)
    datasets.Dataset.from_list(rows).to_parquet(path)
    print(f"wrote {len(rows):>6} rows -> {name}")
    return path

write_parquet(build_clean_rows(easy_idx), "train_easy_clean.parquet")
write_parquet(build_clean_rows(hard_idx), "train_hard_clean.parquet")
write_parquet(build_mixed_rows(easy_idx), "train_easy_mixed.parquet")
write_parquet(build_mixed_rows(hard_idx), "train_hard_mixed.parquet")

# Clean test split (shared validation set for both tracks).
test_rows = []
for i, ex in enumerate(test_raw):
    gt = extract_solution(ex["answer"])
    test_rows.append(build_row(ex["question"], gt, "test", i, ex["answer"], ex["question"]))
write_parquet(test_rows, "test.parquet")

## 5. Spot-check: easy vs hard questions + base model answers

Eyeball a few examples from each bucket to confirm the split is sensible before training. If many "hard" problems look trivially solvable, the 512-token budget may be truncating reasoning rather than reflecting true difficulty.

In [ ]:
N_SHOW = 5

outputs_df = pd.read_parquet(outputs_path) if outputs_path else None

def first_response(idx):
    if outputs_df is None or idx >= len(outputs_df):
        return "<no outputs parquet>"
    resps = outputs_df.iloc[idx]["responses"]
    return resps[0] if len(resps) else "<empty>"

def show_bucket(name, indices):
    print("=" * 100)
    print(f"{name.upper()} SAMPLES")
    print("=" * 100)
    sample = sorted(random.Random(SEED).sample(sorted(indices), min(N_SHOW, len(indices))))
    for idx in sample:
        ex = train_raw[int(idx)]
        acc = per_problem.loc[per_problem["problem_idx"] == idx, "accuracy"].values
        acc = float(acc[0]) if len(acc) else float("nan")
        print(f"\n[idx={idx}  acc={acc:.2f}]")
        print("Q:", ex["question"][:400])
        print("GT:", extract_solution(ex["answer"]))
        print("Model:", str(first_response(int(idx)))[:600])

show_bucket("easy", easy_idx)
show_bucket("hard", hard_idx)

## 6. Next step

Once the split looks good, upload the parquets to the Volume:
```bash
modal run scripts/data_upload_e3_gsm8k.py --local-dir scripts/e3-grpo-gsm8k/data_e3_gsm8k
```